**UID:**
Ian Fuller 117923346

# **CMSC426 Project 3: From Pixels to 3D Worlds : Two-View 3D Reconstruction**

# **Introduction**

We've spent a lot of time working with images, mostly in 2D scenes. Remember Project 1, where we stitched images together by finding common features between them? Well, now it’s time to take things up a notch! Imagine being able to create a 3D view of a scene using just two images and figuring out exactly where the camera was positioned relative to that scene. Sounds exciting, right?

This approach is actually a simplified version of something called Structure from Motion (SfM). While SfM usually involves reconstructing 3D structures from many images taken from different angles, we’ll start with something more manageable: Two-View Reconstruction. Instead of dealing with a whole collection of viewpoints, we’ll be focusing on building a 3D structure using only two images taken from different perspectives. The key idea? Matching features between these two images to reconstruct the scene in 3D and understand how the camera moved between shots.

* Feature Matching
* Outlier Rejection using RANSAC & Estimating the Fundamental Matrix
*  Estimating the Essential Matrix from the Fundamental Matrix
*  Estimating Camera Pose from the Essential Matrix
* Checking for Cheirality Condition using Triangulation
* Linear Triangulation of Points

Reading Module : https://cmsc733.github.io/2022/proj/p3/

Video Lecture : [link](https://drive.google.com/file/d/1KxBYehdtbVpH4A0vsNwbLpuLsGNqYH8F/view?usp=sharing)

Allowed functions: Any functions regarding reading, writing and displaying/plotting images in cv2, matplotlib

Basic math utitlies including convolution operations in numpy and math

# **Step 1: Feature Matching** [5 points]

In [ ]:
# Download data from Google Drive
import gdown

gdown.download_folder(
    id="1ROQl0NTBnxMjLrO5qISyxcTCJUPSMsRS", quiet=True, use_cookies=False
)

In [ ]:
def display_matches(img1, img2, pts1, pts2):

    # This is Chris' code
    dmatches = []

    kp1 = []
    for x, y in pts1:
        kp1.append(cv2.KeyPoint(x.astype(float), y.astype(float), 1))

    # convert corners into key point objects
    kp2 = []
    for x, y in pts2:
        kp2.append(cv2.KeyPoint(x.astype(float), y.astype(float), 1))

    for i in range(len(pts1)):
        dmatches.append(cv2.DMatch(_queryIdx=i, _trainIdx=i, _distance=1))

    out = cv2.drawMatches(
        cv2.cvtColor(img1, cv2.COLOR_BGR2RGB),
        kp1,
        cv2.cvtColor(img2, cv2.COLOR_BGR2RGB),
        kp2,
        dmatches,
        None,
    )

    plt.figure(figsize=(16, 12))
    plt.imshow(out)
    plt.axis("off")
    plt.show()
    """
  Visualize the matching points between two images.

  Input:
      img1: Image 1 in numpy array format.
      img2: Image 2 in numpy array format.
      pts1: Matched points in Image 1 (numpy array of shape Nx2).
      pts2: Matched points in Image 2 (numpy array of shape Nx2).
  """

In [ ]:
def load_data(image1_path, image2_path, npz_path):
    # Load images
    img1 = cv2.imread(image1_path)
    img2 = cv2.imread(image2_path)

    # Load correspondences
    data = np.load(npz_path)
    pts1 = data["pts1"]  # Nx2 array of points in the first image
    pts2 = data["pts2"]  # Nx2 array of points in the second image
    return img1, img2, pts1, pts2

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# File paths
image1_path = "/content/project3/image1.jpg"
image2_path = "/content/project3/image2.jpg"
npz_path = "/content/project3/feature_points.npz"
intrinsics = np.load("/content/project3/intrinsics.npz")
K1 = intrinsics["K1"]
K2 = intrinsics["K2"]
# Load and display matches
img1, img2, pts1, pts2 = load_data(image1_path, image2_path, npz_path)

display_matches(img1, img2, pts1, pts2)

<img src = "https://drive.google.com/uc?export=view&id=1wD6L8Zl2FM75HLn8sy9ub-13ae4Jz0Hk">

# **Step 2: Fundamental Matrix Estimation and RANSAC** [15 points]
<img src="https://cmsc733.github.io/assets/2019/p3/ransac.png">

In [ ]:
def estimate_Fmatrix(img1_pts, img2_pts):

    A = np.empty([8, 9], dtype=float)
    for rowIndex in range(8):

        row = np.array(
            [
                img1_pts[rowIndex][0] * img2_pts[rowIndex][0],
                img1_pts[rowIndex][0] * img2_pts[rowIndex][1],
                img1_pts[rowIndex][0],
                img1_pts[rowIndex][1] * img2_pts[rowIndex][0],
                img1_pts[rowIndex][1] * img2_pts[rowIndex][1],
                img1_pts[rowIndex][1],
                img2_pts[rowIndex][0],
                img2_pts[rowIndex][1],
                1,
            ]
        )
        A[rowIndex] = row
    U, S, V = np.linalg.svd(A)

    F = V[:, 8].reshape(3, 3)

    U, S, V = np.linalg.svd(F)

    S[2] = 0
    F = U @ np.diag(S) @ V

    return F
    """
  Estimate the Fundamental Matrix using matched points from two images.

  Input:
      img1_pts: Matched points from Image 1 (numpy array of shape Nx2).
      img2_pts: Matched points from Image 2 (numpy array of shape Nx2).

  Output:
      F: Estimated Fundamental Matrix (3x3 numpy array).
  """

In [ ]:
def ransac(pts1, pts2, iterations=5000, threshold=1.0):
    # Chris' code
    best_inliers = set()  # holds indicies not the actual
    best_F = None
    for i in range(iterations):
        choices1 = []
        choices2 = []
        for k in range(8):
            k = np.random.randint(0, len(pts1))
            choices1.append(pts1[k])
            choices2.append(pts2[k])
        F = estimate_Fmatrix(choices1, choices2)

        inliers = set()
        for j in range(len(pts1)):
            # pt1 = pts1[j]
            # pt2 = pts2[j]
            pt1 = np.append(pts1[j], 1)
            pt2 = np.append(pts2[j], 1)

            # print(f"RANSAC: pt2.T: {pt2.T}")
            # print(f"RANSAC: F: {F}")
            # print(f"RANSAC: pt1: {pt1}")
            # print(f"RANSAC -> np.abs(pt2.T @ F @ pt1): {np.abs(pt2.T @ F @ pt1)}")
            if np.abs(pt2.T @ F @ pt1) < threshold:
                inliers.add(j)

        # print(f"len(inliers) {len(inliers)}")
        if len(inliers) > len(best_inliers):
            best_inliers = inliers
            best_F = F

    img1_inliers = []
    img2_inliers = []
    for i in best_inliers:
        img1_inliers.append(pts1[i])
        img2_inliers.append(pts2[i])

    return img1_inliers, img2_inliers, best_F
    """
  Apply RANSAC to estimate the Fundamental Matrix robustly.

  Input:
      pts1: Points from Image 1 (numpy array of shape Nx2).
      pts2: Points from Image 2 (numpy array of shape Nx2).
      iterations: Number of RANSAC iterations (int).
      threshold: Distance threshold

  Output:
      img1_inliers: Inliers from Image 1 (numpy array of shape Mx2).
      img2_inliers: Inliers from Image 2 (numpy array of shape Mx2).
      best_F: Best Fundamental Matrix found (3x3 numpy array).
    """

In [ ]:
img1_points, img2_points, F = ransac(pts1, pts2)
print("Fundamental Matrix:", F)
print(f"rank(F) {np.linalg.matrix_rank(F)}")
print(f"det(F) {np.linalg.det(F)}")
print(f"num_inliers: {len(img1_points)}")
# img1, img2,pts1, pts2 = load_data(image1_path, image2_path,npz_path)
# estimate_Fmatrix(pts1, pts2)

display_matches(img1, img2, img1_points, img2_points)

# **Draw epipolar lines** [10 points]

In [ ]:
def compute_epipolar_lines_manual(pts, F):
    lines = np.empty([len(pts), 3], dtype=float)

    for i, pt in enumerate(pts):
        pt = np.append(pt, 1)
        lines[i] = F @ pt.T

    return lines
    """
    Compute epipolar lines for given points using the Fundamental Matrix.

    Input:
        pts: Points in homogeneous coordinates (numpy array of shape Nx2).
        F: Fundamental Matrix (3x3 numpy array).

    Output:
        lines: Epipolar lines in homogeneous form (Nx3 numpy array).

    Comments:
        - Converts points to homogeneous coordinates.
        - Computes lines using the Fundamental Matrix.
    """


# Function to draw epipolar lines on images
def draw_epipolar_lines_img1_points_img2_lines(img1, img2, pts1, F):
    # Chris' code
    img1 = img1.copy()
    img2 = img2.copy()

    for pt in pts1:
        x, y = int(pt[0]), int(pt[1])
        cv2.circle(img1, (x, y), 3, (0, 0, 255), -1)

    lines = compute_epipolar_lines_manual(pts1, F)

    for line in lines:
        x0 = 0
        y0 = int(-(line[0] * 0 + line[2]) / line[1])

        x1 = img2.shape[1]
        y1 = int(-(line[0] * img2.shape[1] + line[2]) / line[1])

        cv2.line(img2, (x0, y0), (x1, y1), (0, 0, 0), 3)

    fig, axes = plt.subplots(1, 2, figsize=(12, 8))
    axes[0].imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Image 1")
    axes[1].imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Image 2")
    plt.show()
    """
  Draw epipolar lines on the second image for points from the first image.

  Input:
      img1: Image 1 in numpy array format.
      img2: Image 2 in numpy array format.
      pts1: Points in Image 1 (numpy array of shape Nx2).
      F: Fundamental Matrix (3x3 numpy array).

  """

In [ ]:
# Draw epipolar lines based on the fundamental matrix
draw_epipolar_lines_img1_points_img2_lines(img1, img2, img1_points, F)

<img src = "https://drive.google.com/uc?export=view&id=1dx0RYxKdiYxuajkIbJhCaVPbYuTjM7Ss">

# **Step 3: Estimate Essential Matrix E** [15 points]

In [ ]:
def estimate_Essentialmatrix(K1, K2, F):

    return K2.T @ F @ K1
    """
  Estimate the Essential Matrix using camera intrinsics and the Fundamental Matrix.

  Input:
      K1: Intrinsic matrix of Camera 1 (3x3 numpy array).
      K2: Intrinsic matrix of Camera 2 (3x3 numpy array).
      F: Fundamental Matrix (3x3 numpy array).

  Output:
      E: Estimated Essential Matrix (3x3 numpy array).

  Comments:
      - Computes the Essential Matrix as E = K2.T * F * K
  """

In [ ]:
E = estimate_Essentialmatrix(K1, K2, F)
print("Essential Matrix E:", E)

# **Step 4: Extracting Poses form E matrix** [20 points]
<img src = "https://drive.google.com/uc?export=view&id=1vizqzdm0gB4sRWQSrvZZPjwezW2OG5tE">

In [ ]:
def get_RTset(E):

    # Andre's code
    # SVD decomposition of the Essential Matrix
    U, _, Vt = np.linalg.svd(E)

    # Define W matrix as in the image
    W = np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]])

    # Compute the four possible rotations and translations
    R1 = U @ W @ Vt
    R2 = U @ W @ Vt
    R3 = U @ W.T @ Vt
    R4 = U @ W.T @ Vt
    print("Determinant for R1", np.linalg.det(R1))
    print("Determinant for R2", np.linalg.det(R2))
    print("Determinant for R3", np.linalg.det(R3))
    print("Determinant for R4", np.linalg.det(R4))

    # Possible translation vectors
    C1 = U[:, 2]
    C2 = -U[:, 2]
    C3 = U[:, 2]
    C4 = -U[:, 2]
    print("C1", C1)
    print("C2", C2)
    print("C3", C3)
    print("C4", C4)

    # Ensure det(R) = 1 for valid rotations
    if np.linalg.det(R1) < 0:
        R1 = -R1
        C1 = -C1
    if np.linalg.det(R2) < 0:
        R2 = -R2
        C2 = -C2
    if np.linalg.det(R3) < 0:
        R3 = -R3
        C3 = -C3
    if np.linalg.det(R4) < 0:
        R4 = -R4
        C4 = -C4
    # Package results
    R_set = [R1, R2, R3, R4]
    T_set = [C1.reshape(3, 1), C2.reshape(3, 1), C3.reshape(3, 1), C4.reshape(3, 1)]

    return R_set, T_set

    """
    Extract possible sets of rotation (R) and translation (T) from the Essential Matrix.

    Input:
        E: Essential Matrix (3x3 numpy array).

    Output:
        R: List of possible rotation matrices (4x 3x3 numpy arrays).
        T: List of possible translation vectors (4x 3x1 numpy arrays).

    Comments:
        - Uses SVD decomposition of the Essential Matrix.
        - Constructs four possible (R, T) pairs and ensures valid rotations

"""

In [ ]:
R_set, T_set = get_RTset(E)
print("Possible Rotations (R):", R_set)
print("Possible Translations (T):", T_set)

# **Step 5: Linear Triangulation** [15 points]

In [ ]:
def linear_triangulation(R_Set, T_Set, pt1, pt2, k):

    points_3d_set = []
    num_points = len(pt1)

    for R, T in zip(R_Set, T_Set):
        # Compute projection matrices P1 and P2
        P1 = k @ np.hstack((np.identity(3), np.zeros((3, 1))))
        P2 = k @ np.hstack((R, T))

        # Triangulate each point pair
        points_3d = []
        for i in range(num_points):
            # Construct the linear system Ax = 0 for triangulation
            x1, y1 = pt1[i]
            x2, y2 = pt2[i]

            A = np.zeros((4, 4))
            A[0] = x1 * P1[2] - P1[0]
            A[1] = y1 * P1[2] - P1[1]
            A[2] = x2 * P2[2] - P2[0]
            A[3] = y2 * P2[2] - P2[1]

            # Solve the system using SVD
            _, _, Vt = np.linalg.svd(A)
            X = Vt[-1]  # The solution is the last row of V^T
            X = X / X[3]  # Convert from homogeneous to 3D coordinates

            points_3d.append(X[:3])

        points_3d_set.append(
            np.array(points_3d)
        )  # Store 3D points for this (R, T) pair

    return points_3d_set
    """
    Perform linear triangulation to estimate 3D points.

    Input:
        R_Set: List of possible rotation matrices (4x 3x3 numpy arrays).
        T_Set: List of possible translation vectors (4x 3x1 numpy arrays).
        pt1: Points in Image 1 (numpy array of shape Nx2).
        pt2: Points in Image 2 (numpy array of shape Nx2).
        k: Camera intrinsic matrix (3x3 numpy array).

    Output:
        points_3d_set: List of 3D points for each (R, T) pair.

    Comments:
        - Triangulates 3D points for each (R, T) pair.
    """

In [ ]:
print("3D points:", linear_triangulation(R_set, T_set, pts1, pts2, K1))

# **Step 6: Cheriality condition** [20 points]

In [ ]:
# Function to select the best pose (R, T) based on the chirality condition
def extract_pose(R_set, T_set, pts_3d_set):
    best_count = -1
    best_index = -1
    R_best = None
    T_best = None
    X_best = None

    for i, (R, T, pts_3d) in enumerate(zip(R_set, T_set, pts_3d_set)):
        pts_cam1 = pts_3d[:, 2] > 0  # Z coordinate is positive for Camera 1
        pts_cam2 = (R @ pts_3d.T + T).T
        pts_cam2_valid = pts_cam2[:, 2] > 0  # And Z is positive for Camera 2

        count = np.sum(pts_cam1 & pts_cam2_valid)

        if count > best_count:
            best_count = count
            best_index = i
            R_best = R
            T_best = T
            X_best = pts_3d

    return R_best, T_best, X_best, best_index
    """
    Select the best pose (R, T) that satisfies the chirality condition.

    Input:
        R_set: List of possible rotation matrices (4x 3x3 numpy arrays).
        T_set: List of possible translation vectors (4x 3x1 numpy arrays).
        pts_3d_set: List of 3D point sets for each (R, T) pair.

    Output:
        R_best: Best rotation matrix (3x3 numpy array).
        T_best: Best translation vector (3x1 numpy array).
        X_best: Best 3D points (Nx3 numpy array).
        index: Index of the best (R, T) pair.

    """

In [ ]:
def compute_cheriality(pt, r3, t):
    depth = (r3 @ pt.T + t).flatten()  # Depth for each point

    # Count points with positive depth
    count_depth = np.sum(depth > 0)

    return count_depth
    """
    Compute the chirality condition to determine if points are in front of the camera.

    Input:
        pt: 3D points (Nx3 numpy array).
        r3: Third row of the rotation matrix (1x3 numpy array).
        t: Translation vector (3x1 numpy array).

    Output:
        count_depth: Number of points with positive depth.

    """

In [ ]:
# Linear Triangulation
point3D_set = linear_triangulation(R_set, T_set, img1_points, img2_points, K1)

# TO-DO: Plot all poses with 3D points as shown in figure
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

colors = ["r", "b", "g", "y"]  # Color for each camera pose and its 3D points
labels = ["cam0", "cam1", "cam2", "cam3"]

for i, (R, T, points_3d) in enumerate(zip(R_set, T_set, point3D_set)):
    # Plot the 3D points for this pose
    points_3d = np.array(points_3d)
    ax.scatter(
        points_3d[:, 0],
        points_3d[:, 1],
        points_3d[:, 2],
        s=1,
        c=colors[i],
        label=labels[i],
    )

    # Plot the camera center
    camera_center = -R.T @ T.flatten()  # Convert T to a 1D array
    ax.scatter(
        camera_center[0],
        camera_center[1],
        camera_center[2],
        c=colors[i],
        marker="^",
        s=100,
    )

    # Draw camera orientation axes
    scale = 2  # Scaling factor for axes
    x_axis = camera_center + scale * R.T[:, 0]  # X-axis in world coordinates
    y_axis = camera_center + scale * R.T[:, 1]  # Y-axis in world coordinates
    z_axis = camera_center + scale * R.T[:, 2]  # Z-axis in world coordinates

    ax.plot(
        [camera_center[0], x_axis[0]],
        [camera_center[1], x_axis[1]],
        [camera_center[2], x_axis[2]],
        "r",
    )
    ax.plot(
        [camera_center[0], y_axis[0]],
        [camera_center[1], y_axis[1]],
        [camera_center[2], y_axis[2]],
        "g",
    )
    ax.plot(
        [camera_center[0], z_axis[0]],
        [camera_center[1], z_axis[1]],
        [camera_center[2], z_axis[2]],
        "b",
    )

# Set plot labels
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.legend()
plt.title("Camera Poses with 3D Points")
plt.show()

<img src = "https://drive.google.com/uc?export=view&id=186Z0ei48e8SV-lEajldKhtilU4r_vdQI">

In [ ]:
# Get pose of camera using chirality condition
R_best, T_best, X_, index = extract_pose(R_set, T_set, point3D_set)


# TO-DO: Plot 3D point cloud with RGB color as shown in figure

<img src = "https://drive.google.com/uc?export=view&id=1KH5L6sUD0d291MAt9OZ2zHecIinTUoXR">

# **Extra Credit: Reconstruct Your Own Scene! (30 Points)**

For this part, you will run an off-the-shelf incremental SfM toolbox such as [COLMAP](https://github.com/colmap/pycolmap) on your own captured multi-view images. Please submit a GIF of the reconstructed 3D structure and the location of the cameras.

For this reconstruction, you can choose your own data. This data could either be a sequence with rigid objects (e.g., a mug or a vase from your vicinity) or any scene you wish to reconstruct in 3D.


Submission:

A gif to visualize the reconstruction of the scene and location of cameras.
<img src = "https://drive.google.com/uc?export=view&id=1MmeeYX77xcQlDVzsDRWyusD8XMO2AY-m">

## Report
You will be graded primarily based on your report.
A demonstration of understanding of the concepts involved in the project are required show the output produced by your code.

Include visualizations of the output of each stage in your pipeline (as shown in the system diagram on page 2), and a description of what you did for each step. Assume that we’re familiar with the project, so you don’t need to spend time repeating what’s already in the course notes. Instead, focus on any interesting problems you encountered and/or solutions you implemented.

# Submission Guidelines

**If your submission does not comply with the following guidelines, you’ll be given ZERO credit.**

Your submission on ELMS(Canvas) must be a pdf file, following the naming convention **YourDirectoryID_proj3.pdf**. For example, xyz123_proj3.pdf.

**All your results and report should be included in this notebook. After you finished all, please export the notebook as a pdf file and submit it to ELMS(Canvas).**

# Collaboration Policy
You are encouraged to discuss the ideas with your peers. However, the code should be your own, and should be the result of you exercising your own understanding of it. If you reference anyone else’s code in writing your project, you must properly cite it in your code (in comments) and your writeup. For the full honor code refer to the CMSC426 Fall 2023 website.

Project Inspiration Credit to CMSC733 COmputer Vision and 16-822: Geometry-based Methods in Vision